# More
## 1. Heteroscedasticity as Signal
Most people test normality and stop. Experts ask: does the variance of X differ across classes?
Run Levene's test for each feature. A feature with equal means but unequal variances across classes is still a predictor — standard r_pb will show near-zero and you'll discard it. This is a systematic blind spot in Phase 2.
Use Welch's t-test (unequal variance) regardless — never Student's t for this kind of data.

## 2. Transformation Search Before Correlation
r_pb measures linear association. Before concluding a feature is weak, test whether a transformation reveals a hidden linear relationship:

log(x), sqrt(x), 1/x, x², Box-Cox (λ estimated from data)
Compute r_pb on each transformed version
If transformed r >> raw r, you have a nonlinear predictor masquerading as noise

The expert move: rank features by max(|r_pb|) across a transformation family, not just raw r.

## 3. Subgroup Non-Stationarity
A feature may be a strong predictor in one stratum and anti-correlated in another — the aggregate r cancels to zero. This is Simpson's Paradox and it's more common than people admit.
For every feature, stratify by each categorical covariate and recompute r_pb per stratum. If variance of stratum-level r is high, the aggregate number is meaningless.

## 4. Rank-Based Concordance Instead of Correlation
Swap r_pb for the rank-biserial correlation computed via Mann-Whitney U:
r_rb = 1 - (2U / (n₀ * n₁))
This is distribution-free, robust to outliers, and directly interpretable as: "probability that a random class-1 observation exceeds a random class-0 observation" (AUROC). You get effect size and a discrimination metric in one number — more actionable than r_pb.

## 5. Partial Correlation
Before declaring a feature the best predictor, ask: is its correlation with the target driven by a confounder?
Compute partial r_pb controlling for every other top-k candidate. A feature that drops from r=0.4 to r=0.05 after partialling out feature B is not an independent predictor — it's a proxy.
pythonfrom pingouin import partial_corr
partial_corr handles the matrix inversion and degrees-of-freedom correctly

## 6. Influence Diagnostics on the Correlation Itself
Compute r_pb with each observation left out (leave-one-out). Plot the LOO distribution of r. If a handful of points are responsible for your entire correlation signal, you don't have a finding — you have outliers.
This is the correlation analogue of Cook's distance and almost nobody does it.

## 7. Test for Restriction of Range
If your data was collected under a selection process (e.g. only approved loans, only admitted patients), the observed r_pb is a downward-biased estimate of the true population correlation. Apply Thorndike's Case II correction:
r_corrected = r_obs * (σ_full / σ_restricted) / sqrt(1 - r_obs² + r_obs²*(σ_full/σ_restricted)²)
You need an estimate of σ_full from domain knowledge or a reference dataset. Ignoring this leads to systematic undervaluation of predictors.

## 8. Reliability Correction (Attenuation)
If your features are measurements with known or estimable measurement error (e.g. survey instruments, sensor readings), observed r_pb is attenuated:
r_true = r_obs / sqrt(r_xx * r_yy)
where r_xx is the reliability of X (e.g. Cronbach's α, test-retest r) and r_yy = 1 for a clean binary target. This tells you the ceiling of what's theoretically recoverable with a perfect instrument.